# Séance 1 :  Deep Learning - Introduction à Pytorch 

Les notebooks sont très largement inspirés des cours de **N. Baskiotis et B. Piwowarski**. Ils peuvent être complétés efficacement par les tutoriels *officiels* présents sur le site de pytorch:
https://pytorch.org/tutorials/

Au niveau de la configuration, toutes les installations doivent fonctionner sur Linux et Mac. Pour windows, ça peut marcher avec Anaconda à jour... Mais il est difficile de récupérer les problèmes.

* Aide à la configuration des machines: [lien](https://dac.lip6.fr/master/environnement-deep/)
* Alternative 1 à Windows: installer Ubuntu sous Windows:  [Ubuntu WSL](https://ubuntu.com/wsl)
* Alternative 2: travailler sur Google Colab (il faut un compte gmail + prendre le temps de comprendre comment accéder à des fichers) [Colab](https://colab.research.google.com)

In [29]:
import torch
print("La version de torch est : ",torch.__version__)
print("Le calcul GPU est disponible ? ", torch.cuda.is_available())
# pour les possesseurs de mac M1 avec la dernière version de pytorch:
print("Le calcul GPU est disponible (apple) ? ", torch.backends.mps.is_available())

# activation plus tard dans les TP

import matplotlib.pyplot as plt
import numpy as np
import sklearn

La version de torch est :  2.15.0+cpu
Le calcul GPU est disponible ?  False
Le calcul GPU est disponible (apple) ?  False


In [30]:
## Chargement des données housing (depuis sklearn) et transformation en tensor.
# from sklearn.datasets import load_housing # => removed
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing(data_home="./data/") ## chargement des données

housing_x = torch.tensor(housing['data'],dtype=torch.float) # penser à typer les données pour éliminer les incertitudes
housing_y = torch.tensor(housing['target'],dtype=torch.float)

print("Nombre d'exemples : ",housing_x.size(0), "Dimension : ",housing_x.size(1))
print("Nom des attributs : ", ", ".join(housing['feature_names']))

# affichage des 5 premiers éléments
print(housing_x[:5])



Nombre d'exemples :  20640 Dimension :  8
Nom des attributs :  MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude
tensor([[ 8.3252e+00,  4.1000e+01,  6.9841e+00,  1.0238e+00,  3.2200e+02,
          2.5556e+00,  3.7880e+01, -1.2223e+02],
        [ 8.3014e+00,  2.1000e+01,  6.2381e+00,  9.7188e-01,  2.4010e+03,
          2.1098e+00,  3.7860e+01, -1.2222e+02],
        [ 7.2574e+00,  5.2000e+01,  8.2881e+00,  1.0734e+00,  4.9600e+02,
          2.8023e+00,  3.7850e+01, -1.2224e+02],
        [ 5.6431e+00,  5.2000e+01,  5.8174e+00,  1.0731e+00,  5.5800e+02,
          2.5479e+00,  3.7850e+01, -1.2225e+02],
        [ 3.8462e+00,  5.2000e+01,  6.2819e+00,  1.0811e+00,  5.6500e+02,
          2.1815e+00,  3.7850e+01, -1.2225e+02]])


## A. Architecture modulaires & réseaux de neurones
Dans le framework pytorch (et dans la plupart des frameworks analogues), le module est la brique de base qui permet de construire un réseau de neurones.  Il permet de représenter en particulier :
* une couche du réseau (linéaire : **torch.nn.Linear**, convolution : **torch.nn.convXd**, ...)
* une fonction d'activation (tanh : **torch.nn.Tanh**, sigmoïde : **torch.nn.Sigmoid** , ReLu : **torch.nn.ReLu**, ...)
* une fonction de coût (MSE : **torch.nn.MSELoss**, L1 :  **torch.nn.L1Loss**, CrossEntropy binaire: **torch.BCE**, CrossEntropy : **torch.nn.CrossEntropyLoss**, ...)
* un ensemble de modules : en termes informatique, un module est un conteneur abstrait qui peut contenir d'autres conteneurs) : plusieurs modules peuvent être mis ensemble afin de former un nouveau module plus complexe.


Le fonctionnement est très proche des fonctions que nous avons vu ci-dessus (un module encapsule en fait une fonction de **torch.nn.Function**), mais de manière à gérer automatiquement les paramètres à apprendre. Un module est ainsi muni :
* d'une méthode **forward** qui permet de calculer la sortie du module à partir des entrées
* d'une méthode **backward** qui permet d'effectuer la rétro-propagation (localement).
* tous les paramètres sont automatiquement ajoutés dans une liste interne, accessible par la fonction **.parameters()** du module.

Ci-dessous un exemple de régression linéaire en utilisant les modules.


In [31]:
EPOCHS=10
EPS = 1e-7

Xdim = housing_x.size(1)
## Création d'une couche linéaire de dimension Xdim->1
net = torch.nn.Linear(Xdim, 1) 

## Passe forward du module :  équivalent à net.forward(x)[:10]
print("Sortie du réseau", net(housing_x)[:10])
## affiche la liste des paramètres du modèle
print("Paramètres et noms des paramètres", list(zip(list(net.parameters()))))
print()
print(list(net.named_parameters()))


Sortie du réseau tensor([[ 26.7903],
        [482.4922],
        [ 62.2261],
        [ 75.7422],
        [ 77.1787],
        [ 44.2584],
        [192.1737],
        [205.9018],
        [218.3618],
        [291.4445]], grad_fn=<SliceBackward0>)
Paramètres et noms des paramètres [(Parameter containing:
tensor([[ 0.0854, -0.1945, -0.0498, -0.0479,  0.2172, -0.2530,  0.0820,  0.3088]],
       requires_grad=True),), (Parameter containing:
tensor([-0.2167], requires_grad=True),)]

[('weight', Parameter containing:
tensor([[ 0.0854, -0.1945, -0.0498, -0.0479,  0.2172, -0.2530,  0.0820,  0.3088]],
       requires_grad=True)), ('bias', Parameter containing:
tensor([-0.2167], requires_grad=True))]


In [32]:

## Création d'une fonction de loss aux moindres carrés
mseloss = torch.nn.MSELoss()
## on créé un optimiseur pour le réseau (paramètres w et b), avec un pas de gradient lr
optim = torch.optim.SGD(params=net.parameters(),lr=EPS) 
# Juste pour info, ce n'est pas utile, les paramètres sont déjà initialisés.
net.reset_parameters()

for i in range(EPOCHS):
    loss = mseloss(net(housing_x).view(-1,1),housing_y.view(-1,1))
    print(f"iteration : {i}, loss : {loss}")
    optim.zero_grad()
    loss.backward()
    optim.step()  

iteration : 0, loss : 98176.1015625
iteration : 1, loss : 11872.38671875
iteration : 2, loss : 2177.4306640625
iteration : 3, loss : 1086.4176025390625
iteration : 4, loss : 961.7234497070312
iteration : 5, loss : 945.5628051757812
iteration : 6, loss : 941.5972900390625
iteration : 7, loss : 939.0067749023438
iteration : 8, loss : 936.5764770507812
iteration : 9, loss : 934.1694946289062


## A.1. Création d'un réseau de neurones

Avec ces briques élémentaires, il est très facile de définir un réseau de neurones standard :
* soit en utilisant le conteneur **torch.nn.Sequential** qui permet d'enchaîner séquentiellement plusieurs modules
* soit en définissant à la main un nouveau module.

Ci-dessous un exemple  pour créer un réseau à deux couches linéaires avec une fonction d'activation tanh des deux manières différentes. Vous remarquez qu'il n'y a pas besoin de définir la méthode **backward**, celle-ci est héritée du conteneur abstrait et ne fait qu'appeler séquentiellement en ordre inverse les méthodes **backward** des différents modules. 

In [33]:
EPS = 1e-2
EPOCHS=50

#Réseau à la main (on le refera à la main derriere)
class DeuxCouches(torch.nn.Module):
  def __init__(self, Xdim):
    super(DeuxCouches,self).__init__()
    self.un = torch.nn.Linear(Xdim,5)
    self.act = torch.nn.Tanh()
    self.deux = torch.nn.Linear(5,1)
  def forward(self,x):
    return self.deux(self.act(self.un(x)))

netDeuxCouches = DeuxCouches(Xdim)

mseloss = torch.nn.MSELoss()
    
optim = torch.optim.SGD(params=netDeuxCouches.parameters(),lr=EPS)
for i in range(EPOCHS):
    loss = mseloss(netDeuxCouches(housing_x),housing_y.view(-1,1))
    print(f"iteration : {i}, loss : {loss}")
    optim.zero_grad()
    loss.backward()
    optim.step()
    

iteration : 0, loss : 7.034055709838867
iteration : 1, loss : 5.755948543548584
iteration : 2, loss : 4.764588832855225
iteration : 3, loss : 3.9957051277160645
iteration : 4, loss : 3.3994100093841553
iteration : 5, loss : 2.9369821548461914
iteration : 6, loss : 2.5783770084381104
iteration : 7, loss : 2.3002970218658447
iteration : 8, loss : 2.084686756134033
iteration : 9, loss : 1.917533040046692
iteration : 10, loss : 1.7879481315612793
iteration : 11, loss : 1.6874816417694092
iteration : 12, loss : 1.6095798015594482
iteration : 13, loss : 1.5491633415222168
iteration : 14, loss : 1.5022937059402466
iteration : 15, loss : 1.4659171104431152
iteration : 16, loss : 1.4376654624938965


iteration : 17, loss : 1.415705680847168
iteration : 18, loss : 1.3986220359802246
iteration : 19, loss : 1.3853235244750977
iteration : 20, loss : 1.374967098236084
iteration : 21, loss : 1.3668955564498901
iteration : 22, loss : 1.360594630241394
iteration : 23, loss : 1.355663537979126
iteration : 24, loss : 1.3517924547195435
iteration : 25, loss : 1.3487440347671509
iteration : 26, loss : 1.346337080001831
iteration : 27, loss : 1.3444325923919678
iteration : 28, loss : 1.342923879623413
iteration : 29, loss : 1.3417290449142456
iteration : 30, loss : 1.3407847881317139
iteration : 31, loss : 1.340042233467102
iteration : 32, loss : 1.3394615650177002
iteration : 33, loss : 1.3390096426010132
iteration : 34, loss : 1.3386590480804443
iteration : 35, loss : 1.3383870124816895
iteration : 36, loss : 1.3381754159927368
iteration : 37, loss : 1.338010549545288
iteration : 38, loss : 1.337881326675415
iteration : 39, loss : 1.3377796411514282
iteration : 40, loss : 1.337699055671692
it

In [34]:
#Création d'un réseau à 1 couche cachée avec le module séquentiel (remplace l'objet précédent)
netSeq = torch.nn.Sequential(torch.nn.Linear(Xdim,5),torch.nn.Tanh(),torch.nn.Linear(5,1))
mseloss = torch.nn.MSELoss()    
optim = torch.optim.SGD(params=netSeq.parameters(),lr=EPS) # extraction auto des paramètres :)
for i in range(EPOCHS):
    loss = mseloss(netSeq(housing_x),housing_y.view(-1,1))
    print(f"iteration : {i}, loss : {loss}")
    optim.zero_grad()
    loss.backward()
    optim.step()

iteration : 0, loss : 3.8380110263824463
iteration : 1, loss : 2.1589808464050293
iteration : 2, loss : 1.9216643571853638
iteration : 3, loss : 1.7900571823120117
iteration : 4, loss : 1.6880321502685547
iteration : 5, loss : 1.608946681022644
iteration : 6, loss : 1.5476487874984741
iteration : 7, loss : 1.500141978263855
iteration : 8, loss : 1.4633262157440186
iteration : 9, loss : 1.4347970485687256


iteration : 10, loss : 1.4126904010772705
iteration : 11, loss : 1.39556086063385
iteration : 12, loss : 1.382287859916687
iteration : 13, loss : 1.372003197669983
iteration : 14, loss : 1.3640340566635132
iteration : 15, loss : 1.3578585386276245
iteration : 16, loss : 1.3530731201171875
iteration : 17, loss : 1.3493645191192627
iteration : 18, loss : 1.3464900255203247
iteration : 19, loss : 1.344261884689331
iteration : 20, loss : 1.3425344228744507
iteration : 21, loss : 1.3411951065063477
iteration : 22, loss : 1.3401563167572021
iteration : 23, loss : 1.3393501043319702
iteration : 24, loss : 1.3387244939804077
iteration : 25, loss : 1.3382387161254883
iteration : 26, loss : 1.337861180305481
iteration : 27, loss : 1.3375673294067383
iteration : 28, loss : 1.3373388051986694
iteration : 29, loss : 1.3371602296829224
iteration : 30, loss : 1.337020754814148
iteration : 31, loss : 1.3369115591049194
iteration : 32, loss : 1.3368256092071533
iteration : 33, loss : 1.3367578983306885

## B. Outils annexes

### B.0 tdqm
Afin de rendre les boucles `for` plus élégantes et surtout plus *suivable*, il faut utiliser le package `tdqm`

In [35]:
# avant tdqm
import time
import random

for i in range(10):
    print(i)
    time.sleep(np.random.rand())


0
1
2
3
4
5
6
7
8
9


In [36]:
# après, avec tdqm
from tqdm import tqdm 

for i in tqdm(range(10)):
    # print(i)
    time.sleep(np.random.rand())

100%|██████████| 10/10 [00:05<00:00,  1.85it/s]



### B.1. DataLoader

Pytorch dispose d'un ensemble d'outils qui permettent de simplifier les démarches expérimentales. Nous allons voir en particulier en commençant par
* le DataLoader qui permet de gérer le chargement de données, le partitionement et la constitution d'ensembles de test et d'apprentissage; 

Le <a href=https://pytorch.org/docs/stable/data.html>**DataLoader**</a> et la classe associée <a href=https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset> **Dataset**</a>  permettent en particulier de :
* charger des données
* pré-processer les données
* de gérer les mini-batchs (sous-ensembles sur lequel on effectue une descente de gradient).

La classe **Dataset** est une classe abstraite qui nécessite l'implémentation que d'une seule méthode, ```__getitem__(self,index)``` : elle renvoie le i-ème objet du jeu de données (généralement un couple *(exemple,label)*. 

La classe **TensorDataset** est l'instanciation la plus courante d'un **Dataset**, elle permet de créer un objet **Dataset** à partir d'une liste de tenseurs qui renvoie pour un index $i$ donné le tuple contenant les $i$-èmes ligne de chaque tenseur.

La classe **DataLoader** permet essentiellement de randomiser et de constituer des mini-batchs de façon simple à partir d'une instance de **Dataset**. Chaque mini-batch est constitué d'exemples tirés aléatoirement dans le **Dataset** passé en paramètre et mis bout à bout dans des tenseurs. La méthode ```collate_fn(*args)``` est utilisée pour cela (nous verrons une customization de cette fonction dans une séance ultérieure). C'est ce générateur qui est généralement parcouru lors de l'apprentissage à chaque itération d'optimisation.

Voici un exemple de code pour utiliser le DataLoader : 


In [37]:
from torch.utils.data import DataLoader,TensorDataset

## Création d'un dataset à partir des deux tenseurs d'exemples et de labels
train_data = TensorDataset(housing_x,housing_y)
## On peut indexer et connaitre la longueur d'un dataset
print("DATASET:\n",len(train_data),train_data[5])

## Création d'un DataLoader
## tailles de mini-batch de 16, shuffle=True permet de mélanger les exemples
# loader est un itérateur sur les mini-batchs des données
loader = DataLoader(train_data, batch_size=16,shuffle=True ) # n'hésitez pas à jouer avec les paramètres

#Premier batch (aléatoire) du dataloader : (nb batch = len/batch_size)
print("DATA LOADER:\n",len(iter(loader)),"\n",next(iter(loader))[0].size())


DATASET:
 20640 (tensor([   4.0368,   52.0000,    4.7617,    1.1036,  413.0000,    2.1399,
          37.8500, -122.2500]), tensor(2.6970))
DATA LOADER:
 1290 
 torch.Size([16, 8])


interpréter les dimensions issues du data loader pour être sur que le processus est compris en profondeur

* Combien de fois chaque point est-il vu?
* En combien de calcul traite-t-on l'ensemble des données?
* Ecrire au brouillon l'expression littérale développée de `cumloss`

In [38]:
EPOCHS=10
EPS=1e-4
netSeq = torch.nn.Sequential(torch.nn.Linear(Xdim,5),torch.nn.Tanh(),torch.nn.Linear(5,1))
optim = torch.optim.SGD(params=netSeq.parameters(),lr=EPS)

# La boucle d'apprentissage :
loop = tqdm(range(EPOCHS))
for i in loop:
    cumloss = 0
    # On parcourt tous les exemples par batch de 16 (paramètre batch_size de DataLoader)
    for bx,by in loader:
        loss = mseloss(netSeq(bx).view(-1),by)
        optim.zero_grad()
        loss.backward()
        optim.step()
        cumloss += loss.item() # item pour un scalaire (sinon .data ou detach)
    # old way
    # print(f"iteration : {i}, loss : {cumloss/len(loader)}") # loss sur un batch => diviser pour avoir une grandeur interprétable
    # new way with tqdm => On ajoute le message dans la barre de progression
    loop.set_description(f"loss : {cumloss/len(loader):.4f}")

loss : 1.3321: 100%|██████████| 10/10 [00:11<00:00,  1.18s/it]


### B.2 Checkpointing
Les modèles Deep sont généralement long à apprendre. Afin de ne pas perdre des résultats en cours de calcul, il est fortement recommander de faire du **checkpointing**, c'est-à-dire d'enregistrer des points d'étapes du modèle en cours d'apprentissage pour pouvoir reprendre à n'importe quel moment l'apprentissage du modèle en cas de problème.  Il s'agit en pratique de sauvegarder l'état du modèle et de l'optimisateur (et de tout autre objet qui peut servir lors de l'apprentissage) toutes les n itérations. Toutes les variables d'intérêt sont en général disponibles par la méthode **state_dict()** des modèles et de l'optimiseur. 

En pratique, vous pouvez utilisé un code dérivé de celui ci-dessous.




In [48]:
# Il existe différentes solutions: en voici une
# mais ça marche
# 
import os

def save_state(epoch,model,optim,fichier):
      """ sauvegarde du modèle et de l'état de l'optimiseur dans fichier """
      state = {'epoch' : epoch, 'model_state': model.state_dict(), 'optim_state': optim.state_dict()}
      torch.save(state,fichier) # pas besoin de passer par pickle
 
def load_state(fichier,model,optim):
      """ Si le fichier existe, on charge le modèle et l'optimiseur """
      epoch = 0
      if os.path.isfile(fichier):
          state = torch.load(fichier)
          model.load_state_dict(state['model_state'])
          optim.load_state_dict(state['optim_state'])
          epoch = state['epoch']

      return epoch
 


Exécuter 2 fois les boites ci-dessous:
* Première itération = 50 epochs
* Deuxième itéation... Anticiper le nombre d'époch !

In [51]:
EPOCHS = 50

# construction rapide d'un réseau de neurones de type PMC
netSeq = torch.nn.Sequential(torch.nn.Linear(Xdim,5),torch.nn.Tanh(),torch.nn.Linear(5,1))
optim = torch.optim.SGD(params=netSeq.parameters(),lr=EPS) # extraction auto des paramètres

os.makedirs("checkpoints", exist_ok=True)
fichier = os.path.join("checkpoints", "netSeq.pth")
start_epoch = load_state(fichier,netSeq,optim)
print(start_epoch)


0


In [52]:

for epoch in tqdm(range(start_epoch,EPOCHS)):
    cumloss = 0
    for bx,by in loader:
        loss = mseloss(netSeq(bx).view(-1),by)
        optim.zero_grad()
        loss.backward()
        optim.step()
        cumloss += loss.item()
    if epoch % 10 ==0: save_state(epoch,netSeq,optim,fichier)



100%|██████████| 50/50 [00:56<00:00,  1.12s/it]



### B.3 GPU 
Afin d'utiliser un GPU lors des calculs, il est nécessaire de transférer les données et le modèle sur le GPU par l'intermédiaire de la fonction **to(device)** des tenseurs et des modules.  Il est impossible de faire une opération lorsqu'une partie des tenseurs sont sur GPU et l'autre sur CPU. Il faut que tous les tenseurs et paramètres soient sur le même device ! On doit donc s'assurer que le modèle, les exemples et les labels sont sur GPU pour faire les opérations.

Par ailleurs, on peut connaître le device sur lequel est chargé un tenseur par l'intermédiaire de ```.device``` (mais pas pour un modèle, il faut aller voir les paramètres dans ce cas).

Une manière simple d'utiliser un GPU quand il existe et donc d'avoir un code agnostique est la suivante : 


In [ ]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
# poru les possesseur de mac MXX:
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

## On charge le modèle sur GPU
## A faire avant la déclaration de l'optimiseur, sinon les paramètres optimisés ne seront pas les mêmes! 
## model =  model.to(device) 
loader = DataLoader(TensorDataset(housing_x,housing_y), batch_size=16,shuffle=True ) 

netSeq = torch.nn.Sequential(torch.nn.Linear(Xdim,5),torch.nn.Tanh(),torch.nn.Linear(5,1))
netSeq = netSeq.to(device)
optim = torch.optim.SGD(params=netSeq.parameters(),lr=EPS)

for i,(bx,by) in tqdm(enumerate(loader)):
    ## On charge le batch sur GPU
    bx, by = bx.to(device), by.to(device)
    loss = mseloss(netSeq(bx).view(-1),by)
    optim.zero_grad()
    loss.backward()
    optim.step()
    # if i % 10 ==0: print("batch ",i)


print("Device du mini-batch : ", bx.device)


1290it [00:01, 1126.20it/s]

Device du mini-batch :  cpu



### B.4 TensorBoard

Durant l'apprentissage de vos modèles, il est agréable de visualiser de quelle manière évolue le coût, la précision sur l'ensemble de validation ainsi que d'autres éléments. TensorFlow dispose d'un outil très apprécié, le TensorBoard, qui permet de gérer très facilement de tels affichages. On retrouve tensorboard dans **Pytorch** dans ```torch.utils.tensorboard``` qui permet de faire le pont de pytorch vers cet outil. 

Le principe est le suivant :
* tensorboard fait tourner en fait un serveur web local qui va lire les fichiers de log dans un répertoire local. L'affichage se fait dans votre navigateur à partir d'un lien fourni lors du lancement de tensorboard.
* Les éléments que vous souhaitez visualiser (scalaire, graphes, distributions, histogrammes) sont écrits dans le fichier de log à partir d'un objet **SummaryWriter** .
* la méthode ```add_scalar(tag, valeur, global_step)``` permet de logger une valeur à un step donné, ```add_scalar(tag, tag_scalar_dic, global_step)``` un ensemble de valeurs par l'intermédiaire du dictionnaire ```tag_scalar_dic``` (un regroupement des scalaires est fait en fonction du tag passé, chaque sous-tag séparé par un **/**).

Il existe d'autres méthodes ```add_XXX``` pour visualiser par exemple des images, des histogrammes (cf <a href=https://pytorch.org/docs/stable/tensorboard.html>la doc </a>).

Le code suivant illustre une manière de l'utiliser. 

In [58]:
# SOLUTION 1: lancer tensorboard, ecrire des loss (ou autres indicateurs) dans un fichiers + lancer la visu dans dans un navigateur à part

from pathlib import Path
from IPython.display import display, HTML
from torch.utils.tensorboard import SummaryWriter

import os

# outils avancés de gestion des chemins
BASEPATH = Path("/tmp")
TB_PATH =  BASEPATH / "logs"
TB_PATH.mkdir(parents=True, exist_ok=True)

# usage externe de tensorboard: (1) lancer la commande dans une console; (2) copier-coller l'URL dans un navigateur
display(HTML("<h2>Informations</h2><div>Pour visualiser les logs, tapez la commande : </div>"))
print(f"tensorboard --logdir {Path(TB_PATH).absolute()}")
print("Une fois effectué, copier-coller l'URL dans votre navigateur pour avoir les courbes d'apprentissage")

tensorboard --logdir c:\tmp\logs
Une fois effectué, copier-coller l'URL dans votre navigateur pour avoir les courbes d'apprentissage


In [ ]:
# SOLUTION 2: Tensorbord à l'interieur du notebook (simple mais pas le plus pratique)

# Solution moins intéressante: ne décommenter qu'en cas d'échec de la première solution !!


# # Spécial notebook, les commandes suivantes permettent de lancer tensorboard
# # En dehors du notebook, il faut le lancer à la main dans le shell : 
# # tensorboard --logdir logs
# %load_ext tensorboard
# %tensorboard --logdir /tmp/logs
# from torch.utils.tensorboard import SummaryWriter

# # Spécial notebook : pour avoir les courbes qui s'affichent dans le notebook, 
# # sinon aller à l'adresse web local indiquée lors du lancement de tensorboard
# from tensorboard import notebook
# notebook.display() # A voir si vous avez une autre fenêtre de gestion de tensorboard ou si vous le voulez à la suite


In [59]:

EPS = 1e-5
EPOCHS=10
netSeq = torch.nn.Sequential(torch.nn.Linear(Xdim,5),torch.nn.Tanh(),torch.nn.Linear(5,1))
netDeuxCouches = DeuxCouches(Xdim)
netSeq.name = "Sequentiel" # nommer les modèles
netDeuxCouches.name = "DeuxCouches"

mseloss = torch.nn.MSELoss()
for model in [netSeq, netDeuxCouches]:
    ## Obtention d'un SummaryWriter
    ## meme répertoire que la commande %tensorboard --logdir logs 
    # ATTENTION AUX noms de fichier sous windows:
    # summary = SummaryWriter(f"/tmp/logs/test/{model.name}/".replace(":","_"))
    summary = SummaryWriter(f"/tmp/logs/test/{model.name}/") # on peut ajouter un timestamp ou des paramètres

    optim = torch.optim.SGD(params=model.parameters(),lr=EPS) 
    for i in tqdm(range(EPOCHS)):
        cumloss = 0
        for bx, by in loader:
            loss = mseloss(model(housing_x),housing_y.view(-1,1))
            optim.zero_grad()
            loss.backward()
            optim.step()  
            cumloss+= loss.item()
        summary.add_scalar(f"loss",cumloss,i) # c'est ici qu'on fait le lien


100%|██████████| 10/10 [00:29<00:00,  2.97s/it]


## B.5 Les fonctions lambda (et l'encodage one-hot)

2 choses distinctes: 

1. Les fonctions `lambda` sont très courantes en python, il faut *a minima* savoir les lire
1. L'encodage one-hot est un outil récurrent dans tout le machine-learning... Et le deep-learning

Ex: je veux passer d'un codage de classe en entiers : `[0, 1, 2]` <BR>
à un codage en vecteur: `cl0 = [1, 0, 0]; cl1=[0, 1, 0]; cl2 = [0, 0, 1]`<BR>
voire à un codage en réels `cl0 = [1., 0., 0.]; cl1=[0., 1., 0.]; cl2 = [0., 0., 1.]`


In [ ]:
# lambda: une nouvelle façon de définir une fonction à la volée

# 1. Construire la fonction 2 x^2 + 4
monpoly = lambda x : 2 * x**2 + 4

# 2. Utiliser la fonction
x = [1., 2., 3.]
print([(i, monpoly(i)) for i in x])


In [ ]:
# 3. Autre exemple pour transformer un entier en vecteur one-hot de dimension 3
from torch.nn.functional import one_hot

to_one_hot = lambda x: one_hot(x,3).float()

# on est dans torch => il faut donner des structures torch
print(to_one_hot(torch.tensor(0)))
print(to_one_hot(torch.tensor(1)))

# ATTENTION aux types: le code suivant provoque une erreur
try:
    print(to_one_hot(2))
except Exception as e:
    print(f"Une erreur s'est produite : {e}")

# => Les fonctions torch ne marchent que sur des structures torch 
# contrairement aux fonctions numpy qui marchent (de plus en plus) sur des listes python

tensor([1., 0., 0.])
tensor([0., 1., 0.])
Une erreur s'est produite : one_hot(): argument 'input' (position 1) must be Tensor, not int


# Construction du sujet à partir de la correction

In [ ]:
###  TODO )"," TODO ",\
    txt, flags=re.DOTALL))
f2.close()

### </CORRECTION> ###